**Name:** CUI Zhaonan, WANG Siming

**EIDs:** 58532227, 58530427

# CS5495 - Course Project (2026A)

## Goal

For the course project, you will apply XAI techniques to analyze a black-box ML model trained on a specific domain or task. 

## Which Task and Model?

You are free to choose the domain and task that is related to your own research interests. 

For models, you should use existing trained model(s), i.e., do not waste time to train your own models (that's a different course!). There are many models available online, and here are two good sources:

- https://www.kaggle.com/models
- https://huggingface.co/

## Research Question

During your analysis, here are some suggested questions to focus on:

- Which XAI method is most faithful to the model?
- How can XAI help to explain the model's inner workings?
- Is there any insight gained about the domain/task from analyzing the model?
- Does the model behave similarly to how a human would do the task?
- What are the failure modes of the model, and how does XAI help to explain why it fails?
- Based on the XAI results, what improvements to the model would you suggest?

## Which XAI Methods?
You may use any XAI to explain and interpret your model to answer the research question. Besides the methods introduced in lecture/tutorial, you are encouraged to use other XAI methods not introduced in class.

You should evaluate the faithfulness of the XAI methods that you try, and select the best XAI method before doing any analysis about how the model works. 

If you want to compare with humans, it will be more convincing to use data about how humans do the task (e.g., eye gaze data).

## Project Proposal

Before actually doing the project, you need to write a **project proposal** so that we can make sure the project is doable within the 3-4 weeks. I can also give you some pointers to relevant methods, if necessary.  
- The project proposal should be at most one page with the following contents: 1) an introduction that briefy states the domain/task and which model you will analyze; 2) a precise description of what you plan to do - e.g., What XAI methods you will use? How will you perform the analysis? What dataset you will use (if necessary)?  How will you evaluate your results?
- The goal of the proposal is to work out, in your head, what your project will be. Once the proposal is done, it is just a matter of implementation!
- *You need to submit the project proposal to Canvas 1 week after the Course project is released.*

## Presentation Video
Each project group needs to make a maximum 1 minute short video to showcase the results of your project. The style should be like YouTube Shorts -- try to make the video as compelling as possible, go for a viral video!

See the details in the "Project Presentations" Canvas assignment.

## What to hand in

Your group needs to turn in the following things.

The following files should be uploaded to "Course Project" on Canvas:

1. This ipynb file `CourseProject-2025A.ipynb` with your source code and documentation/report. **You should write about all the various attempts that you make to find a good solution.**  You may separate your code (.py or .ipynb) and report for better organization. However, your documentation should be in the ipynb file.
2. A PDF version of your ipynb file(s).

Other things that need to be turned in:
- Upload your short video following the instructions in the "Project Presentations" assignment on Canvas.  
- Enter the percentage contribution for each project member using the "Project Group Contribution" assignment on Canvas.
- Submit your project proposal to the "Project Proposal" assignment on Canvas. The project proposal is due 1 week after the course project is released.


## Grading
The marks of the assignment are distributed as follows: 
- 5%  - Project proposal
- 25% - Results using various XAI methods.
- 25% - Trying out new XAI methods not introduced in the tutorials/lectures.
- 20% - Intepretation of XAI results to answer the research question.  More points for insightful observations and analysis.
- 10% - Quality of the report.
- 10% - Quality of the presentation video.
- 5% - Results of voting on presentation video. 
- **Late Penalty:** 25 marks will be subtracted for each day late.

## Groups

You may work in groups up to 2 people. To form the group, go to Canvas -> People -> Project Groups

For group members with less than equal contribution, the marks will be assigned according to the following formula:

- Let A%, B% be the percentage contributions for group members Alice and Bob. A%+B%=100%
- Let x be the group project marks.
- Let M = max(A, B). i.e., the maximum contribution.
- The marks for each member will be prorated according to the maximum contribution.
  - For example, the marks for Alice will be: x*A/M, and likewise for other members.



## FAQ
- *Can my project be my recently submitted or soon-to-be submitted paper?* If you plan to just
turn in the results from your paper, then the answer is no. The project cannot be be work
that you have already done. However, your course project can be based on extending your
work. For example, you can try XAI methods introduced in the course on your own models.


# YOUR METHODS HERE

## 0. What this report shows

A hate-speech detector that reads a tweet and an image is usually assumed to use both.
This project audits one checkpoint, `Amirhossein75/clip-vit-base-mmhs150k-mtl`, and asks
a narrower question than whether it is accurate: which modality does it actually read,
and can the explanation methods that claim to show this be trusted?

The checkpoint reproduces its published metrics, so the audit concerns the model as
released. Of seven candidate attribution methods, two clear a pre-registered
faithfulness floor on text (LIME and KernelSHAP, both near six times the floor), and on
the image side one clears it and barely (occlusion). Those three then play a coalition
game over the two modalities on 3,000 samples. Text carries almost all of the decision:
its Shapley value averages +0.047 against +0.0014 for the image, and it dominates on
99.4% of samples. The image is not inert: on 56.5% of samples its contribution is
negative, so the picture often works against the label the text already implies. The
fusion gate does not predict this split (§5), which rules the gate out as a shortcut
explanation. Ablating the units an explanation blames flips the label far more often
than ablating random units (§7), so the attributions are causally live even though the
image's share of the decision is small.

What the image channel contains stays unresolved. Three probes tried to give the small
image contribution a content and all three came back negative: the attribution lands
slightly below an equal-area control inside printed-text boxes (§6), the concept probe
fails its own validity checks (§6), and re-fitting the fusion head with modality dropout
lowers the image's share rather than raising it (§8). Agreement with the human annotators
moves in the pre-registered direction without reaching the threshold: ρ = -0.07 on the
pre-registered hate-relevant subset and -0.05 once the target level is controlled,
against a required -0.20 (§9). Two measurements were added late and change what the
dataset looks like. A 200-image blind OCR gold set puts EasyOCR's detection rate at
70.9% with a 0% false-positive rate, so roughly half the corpus contains readable words
rather than the third the raw coverage suggested (§1). The concept set turns out to have
been built from detectors rather than a human pass; §6 measures it and §11 records the
deviation. Negative results, including one silent bug that would have invalidated §3,
are documented in §11.

In [1]:
import json
import os

import numpy as np
import pandas as pd

REFINE = "refine-logs"
FIG = "figures"


def load(name):
    with open(os.path.join(REFINE, name), encoding="utf-8") as handle:
        return json.load(handle)


def rows(name):
    out = []
    with open(os.path.join(REFINE, name), encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                out.append(json.loads(line))
    return out


print("artefacts on disk:")
for name in sorted(os.listdir(REFINE)):
    if not name.endswith((".json", ".jsonl")) or name.startswith(("r0", "baseline_")):
        continue
    print(f"  {name:32s} {os.path.getsize(os.path.join(REFINE, name)) / 1024:9.1f} KB")

artefacts on disk:
  p0_gate.json                           2.7 KB
  p0_ocr.jsonl                         992.9 KB
  p0_ocr_stats.json                      0.3 KB
  p0_stats.json                          1.1 KB
  p0_text_gold.json                      0.6 KB
  p1_agreement.json                      1.2 KB
  p1_decision.json                       3.6 KB
  p1_faithfulness.jsonl               2088.3 KB
  p1_image_attr.jsonl                 2012.8 KB
  p1_matrix.json                        13.0 KB
  p1_sanity_curves.json                  5.3 KB
  p1_stability.json                      3.4 KB
  p1_text_attr.jsonl                  1143.3 KB
  p1d_image_attr_test.jsonl           3623.0 KB
  p1d_text_attr_test.jsonl            2635.5 KB
  p2_curves.json                      2178.0 KB
  p2_phi.jsonl                        2302.9 KB
  p2_summary.json                        1.5 KB
  p3_concept_audit.json                  1.3 KB
  p3_concept_text.json                   1.7 KB
  p3_concepts.json   

## 1. The audited model, and whether it is the model we think it is

**Checkpoint.** `Amirhossein75/clip-vit-base-mmhs150k-mtl`: CLIP ViT-B/32 towers
(frozen), a 512-d projection per tower, a gated fusion, and five sigmoid heads
(racist, sexist, homophobe, religion, otherhate). The published thresholds are
`[0.3, 0.7, 0.5, 0.3, 0.55]`.

**Data.** MMHS150K test split. Two disjoint samples are drawn from the same frame:

| sample | n | used for |
|---|---|---|
| analysis | 3,000 | modality Shapley, human agreement, deletion curves, failure modes |
| selection | 499 | the seven-method faithfulness comparison (expensive methods) |

Both were drawn from items whose three annotators are all present, so the human
labels support a strict-majority definition and an agreement bucket. All 3,499 images
were downloaded; coverage is complete.

**Gate check before any explanation.** A checkpoint that does not reproduce its own
numbers cannot be audited, so the released metrics were recomputed first and compared
with the model card, with a tolerance of ±0.03 on ROC-AUC and a floor at the published
macro-F1.

| head | ROC-AUC | published | F1 | published |
|---|---|---|---|---|
| racist | 0.7606 | 0.7646 | 0.6872 | 0.6722 |
| sexist | 0.7970 | 0.8097 | 0.5840 | 0.5889 |
| homophobe | 0.8783 | 0.8817 | 0.7251 | 0.7447 |
| religion | 0.6053 | 0.6175 | 0.2157 | 0.2230 |
| otherhate | 0.8381 | 0.8424 | 0.6182 | 0.6161 |
| **macro (5 heads)** | **0.7759** | **0.7832** | **0.5660** | **0.5690** |

Verdict: **REPRODUCED**. macro ROC 0.776 within 0.03 of 0.783 and macro F1 0.566 above the 0.539 floor. No threshold re-tuning was needed
for the gate itself; the published thresholds are kept for every later decision-set
computation, so the explanations describe the released decision rule rather than a
re-calibrated one.

The `religion` head is a different matter: AUC 0.6053
on 2.70% prevalence, with F1
0.2157. It is accurate enough to pass the gate and
too weak to interpret, so it is **excluded from the analysis target**: the target
probability is the mean of the four usable heads (racist, sexist, homophobe,
otherhate), whose macro ROC-AUC is 0.8185 and macro-F1
0.6536. The five-head numbers appear only in this gate table.

**Human labels.** Strict majority non-zero gives a hate rate of
50.0%; "any annotator" gives
74.1%. The gap between the two definitions is
what §9 turns on. Agreement buckets on the analysis set: all annotators agree
932 items, partial agreement
1591, all differ
477.

**Images.** The dataset ships no OCR text for this split
(0.17% coverage), so text inside images was
recovered with EasyOCR at confidence ≥ 0.6: 1279 of 3499
images (36.5% of the 3,499 used here) carry at least one box,
1.79 boxes per image on average, covering
3.2% of the image area. Boxes below the confidence cut account
for 39.7% of detections.

That coverage is a lower bound, and the bound was measured. A blind
gold set of 200 random analysis images, labelled without seeing the
detector output, gives a detection rate of
70.9% (95% CI 61.5%–78.8%)
and a false-positive rate of 0.00%
(95% CI 0.00%–3.85%);
84.9% of images agree, κ = 0.70.

Dividing the observed coverage by the detection rate gives an estimated true share of
51.6% text-bearing images, which matches the
human rate on the gold set (51.8%) to within 0.2 pp. That
changes two things downstream: roughly half the corpus contains readable words, where the
raw coverage had suggested a third; and any group built from EasyOCR detection is a
*subset* of the text-bearing images with essentially no contamination in the other
direction, which §6 relies on. The implied share assumes the gold set's detection rate
transfers to all 3,499 images, which is reasonable because the sample is random from the
same pool; the interval on the detection rate propagates to roughly ±5 pp on the implied
share.

In [2]:
gate = load("p0_gate.json")
table = pd.DataFrame({
    name: {
        "ROC-AUC": block["auc"],
        "published AUC": block["published_auc"],
        "F1": block["f1"],
        "published F1": block["published_f1"],
        "prevalence": block["prevalence"],
    }
    for name, block in gate["per_head"].items()
}).T
print(f"verdict: {gate['verdict']}  (tolerance {gate['tolerance']}, floor {gate['f1_floor']:.3f})")
print(table.round(4).to_string())
print()
print(f"macro 5 heads : AUC {gate['macro_auc_5']:.4f}  F1 {gate['macro_f1_5']:.4f}")
print(f"macro 4 usable: AUC {gate['macro_auc_4_usable']:.4f}  F1 {gate['macro_f1_4_usable']:.4f}")

stats = load("p0_stats.json")
print()
print("analysis split:", stats["main"]["n"], "| strict hate rate", stats["main"]["hate_rate_majority"],
      "| any-annotator", stats["main"]["hate_rate_any_annotator"])
print("agreement buckets:", stats["main"]["agreement_buckets"])

ocr = load("p0_ocr_stats.json")
print()
print(f"OCR: {ocr['with_text']}/{ocr['requested']} images with text "
      f"({ocr['with_text_share']:.3%}), mean {ocr['mean_boxes_per_image']:.2f} boxes, "
      f"{ocr['mean_area_share']:.3%} of area")

verdict: REPRODUCED  (tolerance 0.03, floor 0.539)
           ROC-AUC  published AUC      F1  published F1  prevalence
racist      0.7606         0.7646  0.6872        0.6722      0.3937
sexist      0.7970         0.8097  0.5840        0.5889      0.1747
homophobe   0.8783         0.8817  0.7251        0.7447      0.1220
religion    0.6053         0.6175  0.2157        0.2230      0.0270
otherhate   0.8381         0.8424  0.6182        0.6161      0.2453

macro 5 heads : AUC 0.7759  F1 0.5660
macro 4 usable: AUC 0.8185  F1 0.6536

analysis split: 3000 | strict hate rate 0.4997 | any-annotator 0.741
agreement buckets: {'all_agree': 932, 'partial': 1591, 'all_differ': 477}

OCR: 1279/3499 images with text (36.550%), mean 1.79 boxes, 3.190% of area


## 2. Units, interventions, and what "faithful" is allowed to mean

The numbers in §3 depend on three definitions.

**Units.** Text explanations act on *content BPE tokens*: CLIP's tokenizer adds a
start and an end sentinel, and those two are never units. Image explanations act on
the 7×7 patch grid (32×32 pixels per cell), the resolution Grad-CAM actually has.

**Deletion is an intervention on the mask, not on the ids.** CLIP's text tower pools
at the *first* end-of-sequence token found by argmax, so writing a pad id into a
sentence silently moves the pooled position forward and truncates the tail, an
intervention that changes two things at once. Deletion here sets `attention_mask` to
0 at the chosen positions and leaves `input_ids` untouched. On the image side,
deletion writes the mean colour, which is exactly the "blank" counterfactual used in
the coalition game, so §3 and §5 agree about what removing evidence means.

**The missing-modality counterfactual is content-free but not zero.** Text is replaced
by the encoding of the empty string; the image is replaced by a mean-colour image.
Both are real inputs the towers can process, which matters because a zero vector is
off-manifold for CLIP's LayerNorm.

**Faithfulness.** A ranking is scored by AOPC: delete units in the order the method
ranks them (highest first), measure the drop in the target at each of six fractions,
and average. Insertion mirrors it, starting from the blank modality. The reported
number is the mean of the deletion and insertion AOPC, so a method cannot win by
ranking everything as "important". Every method is compared with a random ordering of
the same units, averaged over repeats, by a paired test on the same samples.

The floors were fixed before the comparison ran: **text AOPC > 0.01, image AOPC >
0.001**. The ten-fold difference is deliberate: image attributions on CLIP-scale
models are numerically much smaller, and a single floor would either pass noise on the
image side or fail every method on the text side. The rule for naming a winner, also
pre-registered: clear the floor and beat random; highest AOPC wins; a difference below
0.001 or a paired p above 0.05 is a tie, broken by comprehensiveness at 20%, then by
seed/noise stability, then in favour of the deterministic method.

In [3]:
import sys
sys.path.insert(0, "src")

import torch
from mmhs.attribution import content_positions

# The intervention, shown on one sentence: masking a token must not re-tokenise.
tok_ids = torch.tensor([[49406, 320, 1125, 49407, 49407, 49407]])
tok_mask = torch.tensor([[1, 1, 1, 1, 0, 0]])
print("content positions (sentinels excluded):", content_positions(tok_mask))
before = tok_ids.clone()
masked = tok_mask.clone()
masked[0, content_positions(tok_mask)[1]] = 0
print("after deleting one token, ids unchanged:", torch.equal(tok_ids, before),
      "| mask:", masked.tolist())

matrix = load("p1_matrix.json")
print()
print("effect floors used below:", 0.01, "(text) /", 0.001, "(image)")
print("fractions scored:", matrix["fractions"])

content positions (sentinels excluded): [1, 2]
after deleting one token, ids unchanged: True | mask: [[1, 1, 0, 1, 0, 0]]

effect floors used below: 0.01 (text) / 0.001 (image)
fractions scored: [0.2, 0.5]


## 3. Seven candidates, one winner per modality

| modality | method | AOPC | comp@20% | suff@20% | paired p vs random |
|---|---|---|---|---|---|
| text | occlusion | +0.0013 | +0.0012 | +0.0038 | 1.25e-04 |
| text | lime | +0.0620 | +0.0643 | +0.0968 | 6.83e-82 |
| text | kernelshap | +0.0606 | +0.0611 | +0.0933 | 4.51e-81 |
| image | occlusion | +0.0013 | +0.0012 | +0.0038 | 1.08e-81 |
| image | gradcam | -0.0010 | -0.0011 | -0.0001 | 9.93e-51 |
| image | ig | -0.0003 | -0.0005 | +0.0012 | 3.58e-17 |
| image | scorecam | -0.0001 | -0.0002 | +0.0017 | 4.31e-01 |
| text | *random order* | +0.0179 | +0.0076 | --- | --- |
| image | *random order* | -0.0001 | -0.0002 | --- | --- |

Paired test: one-sided Wilcoxon on the per-sample difference against the random
ordering of the same units, n = 499 selection samples.

**Text.** LIME wins with AOPC
+0.0620, 6.2× the
0.01 floor; KernelSHAP follows at +0.0606.
The gap between them is 0.0015 with a paired p of
7.8e-08, so it is not a tie by the pre-registered rule, but the
practical reading is that the two surrogate methods agree, and the choice between them
does not carry the argument. Occlusion fails the floor on text: AOPC
+0.0013
against a random ordering at
+0.0179. That is the first genuinely
uncomfortable result in this report: the textbook model-agnostic reference method is
*worse than random* at ranking text tokens on this checkpoint, because single-token
deletions are strongly non-additive here, while LIME and KernelSHAP fit a local
additive surrogate and therefore inherit the model's actual additive structure.

**Image.** Occlusion is the only method above the 0.001 floor
(+0.0013); Grad-CAM, integrated gradients and
Score-CAM are all at or below zero, i.e. deleting the cells they call most important
raises the target slightly more often than deleting random cells. This is itself a
finding about the checkpoint: the vision tower's gradient signal through the fusion
gate is close to uninformative at 7×7 resolution.

**Three caveats that must travel with the winner.** AOPC is LIME's own optimisation
target (it fits weights to reproduce local deletions), so LIME's margin over
KernelSHAP is partly self-fulfilling; the defensible claim is only that both clear the
floor by a wide margin. The image winner clears the floor by
1.29×, not the six-fold margin seen on text, which is
consistent with the image's contribution being small in absolute terms, which §5
confirms independently through a completely different route. And the LIME kernel width
is the library's text default (25 on the Euclidean distance over token presence), not a
pre-registered choice: the tabular default, 0.25·√n, collapses onto near-full coalitions
on this model and returns almost uniform coefficients, which was discovered while
building the method rather than specified in advance. The reported comparison is
therefore "LIME as shipped against the alternatives", not "LIME at its best possible
setting".

In [4]:
matrix = load("p1_matrix.json")
decision = load("p1_decision.json")

for modality in ("text", "image"):
    print(f"--- {modality} ---")
    block = matrix[modality]
    for name, entry in block.items():
        print(f"  {name:11s} AOPC {entry['aopc_del']['mean']:+.4f}  "
              f"comp20 {entry['comp_20']['mean']:+.4f}  suff20 {entry['suff_20']['mean']:+.4f}  "
              f"random {entry['aopc_del']['random_mean']:+.4f}  p {entry['aopc_del']['p_vs_random']:.1e}")
    print(f"  winner: {decision['modalities'][modality]['winner']}")
print()
print("text tie block:", json.dumps(decision["modalities"]["text"]["tie"], indent=1))

--- text ---
  occlusion   AOPC +0.0013  comp20 +0.0012  suff20 +0.0038  random +0.0179  p 1.2e-04
  lime        AOPC +0.0620  comp20 +0.0643  suff20 +0.0968  random +0.0179  p 6.8e-82
  kernelshap  AOPC +0.0606  comp20 +0.0611  suff20 +0.0933  random +0.0179  p 4.5e-81
  winner: lime
--- image ---
  occlusion   AOPC +0.0013  comp20 +0.0012  suff20 +0.0038  random -0.0001  p 1.1e-81
  gradcam     AOPC -0.0010  comp20 -0.0011  suff20 -0.0001  random -0.0001  p 9.9e-51
  ig          AOPC -0.0003  comp20 -0.0005  suff20 +0.0012  random -0.0001  p 3.6e-17
  scorecam    AOPC -0.0001  comp20 -0.0002  suff20 +0.0017  random -0.0001  p 4.3e-01
  winner: occlusion

text tie block: {
 "other": "kernelshap",
 "delta_aopc": 0.0014795662565237622,
 "paired_p": 7.774036963713871e-08,
 "tied": false
}


![Figure 1. AOPC per candidate method, with the random ordering as the dashed line. Error bars are standard errors over the 499 selection samples.](figures/fig1_aopc.png)

*Figure 1. AOPC per candidate method, with the random ordering as the dashed line. Error bars are standard errors over the 499 selection samples.*

## 4. Does the attribution survive a random model?

An explanation that also describes a randomly initialised network is describing the
architecture, not the learned function. Each tower is randomised on its own: the
last 0, 3, 6, 9 or all 12 encoder blocks re-initialised, the other tower and the fusion
head untouched. The attribution is then compared with the trained model's by Spearman
correlation over units. Pre-registered: the correlation must fall
monotonically with depth ($\rho_{\text{depth}} < -0.7$) and collapse at full
randomisation ($r_{\text{full}} < 0.2$). n = 100 selection samples.

| tower | method | depth ρ | full r | gate |
|---|---|---|---|---|
| text | occlusion | -0.90 | +0.06 | pass |
| text | lime | -0.90 | +0.09 | pass |
| text | kernelshap | -0.90 | +0.06 | pass |
| image | occlusion | -1.00 | -0.04 | pass |
| image | gradcam | -0.90 | +0.68 | fail |
| image | ig | -1.00 | +0.33 | fail |
| image | scorecam | -1.00 | +0.23 | fail |

The ordering is the interesting part. At three randomised text blocks, occlusion has
already fallen to 0.57 while LIME and KernelSHAP are
still at 0.90 and 0.95.
Occlusion's ranking depends on individual token deletions, which are exactly what a
partly-randomised tower distorts; the surrogate methods fit many coalitions at once
and are correspondingly more robust. This is the second independent argument against
occlusion on text, and it was measured before the AOPC comparison was scored.
**Stability.** Worst pair over repeats (`p1_stability.json`):

| modality | method | min Spearman | min top-10% overlap | verdict |
|---|---|---|---|---|
| text | lime | 0.66 | 0.80 | pass |
| text | kernelshap | 0.85 | 0.86 | pass |
| image | occlusion | 1.00 | 0.97 | pass |
| image | gradcam | 1.00 | 0.98 | pass |
| image | ig | 1.00 | 0.97 | pass |
| image | scorecam | 0.99 | 0.99 | pass |

In [5]:
curves = load("p1_sanity_curves.json")
print("levels:", curves["levels"])
for modality in ("text", "image"):
    print(f"--- {modality} tower ---")
    for name, block in curves["gates"][modality].items():
        print(f"  {name:11s} rho_depth {block['rho_depth']:+.2f}  r_full {block['r_full']:+.2f}  "
              f"{'pass' if block['passes'] else 'fail'}")

if os.path.exists(os.path.join(REFINE, "p1_stability.json")):
    stability = load("p1_stability.json")
    print()
    for group in ("text_seeds", "image_noise"):
        for name, block in stability.get(group, {}).items():
            print(f"  {group:11s} {name:11s} min rho {block['min_spearman']:+.2f}  "
                  f"min overlap {block['min_top10_overlap']:.2f}")

levels: [0, 3, 6, 9, 12]
--- text tower ---
  occlusion   rho_depth -0.90  r_full +0.06  pass
  lime        rho_depth -0.90  r_full +0.09  pass
  kernelshap  rho_depth -0.90  r_full +0.06  pass
--- image tower ---
  occlusion   rho_depth -1.00  r_full -0.04  pass
  gradcam     rho_depth -0.90  r_full +0.68  fail
  ig          rho_depth -1.00  r_full +0.33  fail
  scorecam    rho_depth -1.00  r_full +0.23  fail

  text_seeds  lime        min rho +0.66  min overlap 0.80
  text_seeds  kernelshap  min rho +0.85  min overlap 0.86
  image_noise occlusion   min rho +1.00  min overlap 0.97
  image_noise gradcam     min rho +1.00  min overlap 0.98
  image_noise ig          min rho +1.00  min overlap 0.97
  image_noise scorecam    min rho +0.99  min overlap 0.99


![Figure 2. Attribution agreement with the trained model as encoder blocks are randomised. A faithful method should fall to zero.](figures/fig4_gates.png)

*Figure 2. Attribution agreement with the trained model as encoder blocks are randomised. A faithful method should fall to zero.*

## 5. How much of the decision is the image? A two-player coalition game

Attribution methods disagree; the coalition game does not, because with two players
the Shapley value has a closed form:

$$\phi_T = \tfrac12\big[v(T)-v(\varnothing)\big] + \tfrac12\big[v(T,I)-v(I)\big], \qquad
\phi_I = \tfrac12\big[v(I)-v(\varnothing)\big] + \tfrac12\big[v(T,I)-v(T)\big]$$

where $v(\varnothing)$ is the blank-text/blank-image input, $v(T)$ is real text over a
mean-colour image, $v(I)$ is empty text over the real image, and $v(T,I)$ is the actual
input. The target is the four-head mean probability on 3,000 analysis samples.
Completeness ($\phi_T + \phi_I = v(T,I) - v(\varnothing)$) holds to
6.9e-08, so the decomposition is exact rather than
approximate.

| quantity | value |
|---|---|
| mean $\phi_T$ | +0.0467 |
| median $\phi_T$ | +0.0152 |
| share of samples with $\phi_T > 0$ | 54.6% |
| mean $\phi_I$ | +0.0014 |
| median $\phi_I$ | -0.0004 |
| share of samples with $\phi_I < 0$ | 56.5% |
| text-dominant samples ($|\phi_T| > |\phi_I|$) | 99.4% |
| interaction $v(T,I)-v(T)-v(I)+v(\varnothing)$ | -0.0007 |

**The text accounts for most of the decision.** Mean $\phi_T$ is
34× the mean $\phi_I$. In a
paired sense the image moves the target by about one percentage point, and that is the
ceiling on any "the model uses the image" claim.

**"Ignored" is the wrong word.** $\phi_I$ is negative on
56.5% of samples: for a majority of inputs, replacing the
picture with a blank one *raises* the hate probability. The image is not a neutral
modality that the fusion gate happens to down-weight; it frequently pulls against the
text's evidence. That asymmetry is invisible in accuracy numbers and invisible in
gate values, and it is the strongest argument for doing the coalition game rather than
reading the gate.

**Per-head, the effect is not uniform.** Mean head probabilities with both modalities
blank versus both present:

| head | blank | both | change |
|---|---|---|---|
| racist | 0.346 | 0.445 | +0.098 |
| sexist | 0.242 | 0.374 | +0.132 |
| homophobe | 0.205 | 0.285 | +0.080 |
| religion | 0.163 | 0.112 | -0.051 |
| otherhate | 0.540 | 0.422 | -0.117 |

The `otherhate` head *drops* when both modalities are present, and `religion`, the
head excluded from the target, moves the most. Neither effect is visible from the
target alone, which is why the per-head table is reported even though the analysis is
target-level.

**Deletion curves agree with the Shapley values.** Removing text tokens in winning-
attribution order costs +0.0463 of target probability;
removing image cells costs +0.0010; removing random
units of either kind costs -0.0032, i.e. nothing.
The gap between text and image, +0.0453, is measured
on a different instrument (deletion from the full input) than the Shapley values
(coalitions from the blank input) and lands on the same conclusion.

In [6]:
phi = load("p2_summary.json")
print(json.dumps({key: phi[key] for key in
                  ("n", "phi_t", "phi_i", "text_dominance_share", "phi_i_negative_share",
                   "interaction_mean", "completeness_max_error", "deletion")}, indent=1))

heads = pd.DataFrame({
    "blank": phi["heads_empty_mean"],
    "both": phi["heads_both_mean"],
}, index=phi["head_names"])
heads["change"] = heads["both"] - heads["blank"]
print()
print(heads.round(3).to_string())

rows = rows("p2_phi.jsonl")
print()
print(f"samples in p2_phi.jsonl: {len(rows)}")
print(f"share phi_T > 0: {np.mean([row['phi_t'] > 0 for row in rows]):.3f}   "
      f"share phi_I < 0: {np.mean([row['phi_i'] < 0 for row in rows]):.3f}")

{
 "n": 3000,
 "phi_t": {
  "mean": 0.0466793792595466,
  "median": 0.015238191932439804,
  "q05": -0.0512226527556777,
  "q95": 0.19589197263121605,
  "share_positive": 0.546
 },
 "phi_i": {
  "mean": 0.0013649850413203239,
  "median": -0.00042062997817993164,
  "q05": -0.0027185031212866306,
  "q95": 0.008344700932502743,
  "share_positive": 0.435
 },
 "text_dominance_share": 0.9943333333333333,
 "phi_i_negative_share": 0.565,
 "interaction_mean": -0.0007081497112909953,
 "completeness_max_error": 6.891787052154541e-08,
 "deletion": {
  "text_reduction": 0.04632282634109459,
  "image_reduction": 0.0010064855417949241,
  "random_reduction": -0.0031722472318066845,
  "text_minus_image": 0.045316340799299665,
  "text_vs_random_p": 1.412494091781504e-74,
  "image_vs_random_p": 1.40478944688315e-276
 }
}

           blank   both  change
racist     0.346  0.445   0.098
sexist     0.242  0.374   0.132
homophobe  0.205  0.285   0.080
religion   0.163  0.112  -0.051
otherhate  0.540  0.422  -

![Figure 3. Per-sample Shapley values for the two modalities. The image panel is clipped to ±0.01 for readability; the mass sits slightly left of zero.](figures/fig2_phi.png)

*Figure 3. Per-sample Shapley values for the two modalities. The image panel is clipped to ±0.01 for readability; the mass sits slightly left of zero.*

![Figure 4. Deletion curves: mean target drop with 95% interval across analysis samples.](figures/fig3_deletion.png)

*Figure 4. Deletion curves: mean target drop with 95% interval across analysis samples.*

## 6. What the image channel contains

Two probes ask what the image contributes on the samples where it does contribute: a
patch-level overlap test against printed text inside the picture, and a concept probe
on the image tower. Both come back negative. The concept probe also fails its own
label audit, which is a separate finding and is reported as such.

**Patch attribution against printed text.** Each 7×7 cell gets a text weight equal to
the fraction of its area covered by an EasyOCR box, so a small box that only clips a
cell still counts and no sample is dropped for having text too small to contain a cell
centre. The control keeps the *union* area of the text boxes fixed and moves a square of
that area to a random position, 100 placements, weighted the same way. The comparison is
therefore "the same amount of image, placed on the text" against "placed elsewhere", not
against zero.

| quantity | value |
|---|---|
| samples with at least one text box | 1,113 of 3,000 |
| mean area share of the boxes | 8.0% |
| mean attribution share inside the text | 7.3% |
| mean share for the equal-area control | 7.8% |
| excess | -0.5 pp |
| paired Wilcoxon p | 2.1e-07 |
| samples above the control's 95th percentile | 7.3% |

The excess is -0.5 pp against a pre-registered threshold of +15 pp,
so the criterion is missed, and it is missed in the
*negative* direction: attribution inside the text boxes is slightly below an equal-area
control, and the paired test rejects equality that way
(p = 2.1e-07). Only 7.3% of images put more attribution
on the text than the control's 95th percentile. On this checkpoint the image tower is
not re-reading the words inside the picture; if anything it looks away from them. A negative
excess does not make text unimportant; it says that a text region draws no more
attribution than any other patch of the same size. The
control square matches the *union* of the text boxes; using the sum of the box areas
instead, which double-counts overlapping boxes and makes the control larger, moves the
excess to -1.1 pp, so the sign and the verdict do not
depend on that choice.

**Do the concept labels hold up?** The concept set was built from three detectors, not
from a human pass (§11). A stratified sample of 100 positives and 100 negatives per
concept was therefore labelled blind, which supports two quantities: sensitivity (share
of cue-positives a human also calls positive) and specificity (share of cue-negatives a
human rejects). Precision is not reported because the stratified sample cannot supply
the population base rate.

| concept | labelled + / − | sensitivity | specificity |
|---|---|---|---|
| flat_graphic | 96 / 99 | 0.71 (0.61–0.79) | 0.65 (0.55–0.73) |
| printed_text | 100 / 97 | 0.93 (0.86–0.97) | 0.94 (0.87–0.97) |
| face | 96 / 93 | 0.83 (0.75–0.89) | 0.56 (0.46–0.66) |

`printed_text` is a usable proxy at both ends. `face` and `flat_graphic` are not: their
negative sets are 44% and 35% contaminated with images that do contain the concept, so a
CAV trained on them is partly fitted to the wrong side. The same labels expose a second
problem: how much of each concept set also contains printed text:

| concept | text share, positives | text share, negatives | gap |
|---|---|---|---|
| printed_text | 94.5% | 5.0% | +89.5 pp |
| face | 52.0% | 44.8% | +7.3 pp |
| flat_graphic | 63.0% | 50.7% | +12.3 pp |

For `face` the two sides are 52.0% and 44.8%, which is almost equality: a "face"
direction is partly a text direction. `flat_graphic` is 63.0% against 50.7%. Only
`printed_text` separates cleanly, and that is the concept whose CAV the probe still
cannot use.

**Concept probe (TCAV).** A logistic CAV is fitted in the image tower's 768-d pooler
space; the sensitivity is the gradient of the four-head target with respect to that
pooler output, dotted with the unit CAV. Twenty random unit directions give the null.

| concept | CAV accuracy | mean cosine | TCAV | p vs null | split-half ρ |
|---|---|---|---|---|---|
| printed_text | 0.623 | -0.112 ± 0.032 | 0.000 | 1.00 | +0.22 |
| face | 0.611 | +0.091 ± 0.026 | 1.000 | 0.00 | +0.72 |
| flat_graphic | 0.857 | -0.115 ± 0.076 | 0.137 | 0.55 | +0.97 |

Both halves of this design fail, independently.

*The statistic is degenerate here.* The per-sample gradient has a mean cosine of
0.830 with its own mean, so its direction barely varies across inputs. A sign
test on a nearly fixed direction can only return 0 or 1, which is what the first two
rows show, and the null betrays the same collapse: 20 random directions produce scores
that jump between 0.0003 and 0.981 (sd 0.383). The near-constancy is
mechanistic. With the four sigmoid heads saturated for most inputs, the derivative's
direction comes from the fusion head's weights; the image input barely moves it.

*The concept directions are weak.* CAV accuracy is 0.62 for `printed_text` and `face`
against balanced 400/400 sets, and 0.86 for `flat_graphic`. The `printed_text` case
rules out bad annotation as the explanation: its labels are 0.93/0.94. The limit sits
in the gradient.

The section's answer is therefore a double negative: no evidence that the image
channel's contribution is text-in-image, and no concept-level claim that survives its
own validity checks.

In [7]:
overlap = load("p3_overlap.json")
print(json.dumps({key: overlap[key] for key in
                  ("n_with_text", "mean_observed_share", "mean_null_share", "mean_excess",
                   "p_wilcoxon", "share_above_null_p95", "mean_text_area_share", "passes")},
                 indent=1))

audit = load("p3_concept_audit.json")
print()
print(pd.DataFrame(audit["per_concept"]).T[
    ["n_positive_labelled", "n_negative_labelled", "sensitivity", "specificity"]].round(3).to_string())

ctext = load("p3_concept_text.json")
print()
print("double-labelled agreement:",
      f"{ctext['double_agreement_raw']:.3f} (kappa {ctext['double_agreement_kappa']:.3f}, "
      f"n={ctext['n_double_usable']}, passes={ctext['passes']})")

tcav = load("p3_tcav.json")
print()
print(f"gradient collinearity: {tcav['sensitivity_collinearity']:.3f}")
print(f"null over {tcav['null']['n']} random directions: "
      f"{tcav['null']['mean']:.3f} +- {tcav['null']['sd']:.3f}")
for name, block in tcav["concepts"].items():
    print(f"  {name:13s} TCAV={block['tcav']:.3f}  CAV acc={block['cav_accuracy']:.3f}  "
          f"cos={block['mean_cosine']:+.3f}+-{block['cosine_sd']:.3f}  p={block['p_null']:.2f}")

{
 "n_with_text": 1113,
 "mean_observed_share": 0.0726686039705713,
 "mean_null_share": 0.0775036471989122,
 "mean_excess": -0.0048350432283409,
 "p_wilcoxon": 2.05022891555532e-07,
 "share_above_null_p95": 0.07277628032345014,
 "mean_text_area_share": 0.08027636301216569,
 "passes": false
}

             n_positive_labelled n_negative_labelled sensitivity specificity
flat_graphic                  96                  99    0.708333    0.646465
printed_text                 100                  97        0.93    0.938144
face                          96                  93    0.833333     0.55914

double-labelled agreement: 0.935 (kappa 0.868, n=92, passes=True)

gradient collinearity: 0.830
null over 20 random directions: 0.397 +- 0.383
  printed_text  TCAV=0.000  CAV acc=0.623  cos=-0.112+-0.032  p=1.00
  face          TCAV=1.000  CAV acc=0.611  cos=+0.091+-0.026  p=0.00
  flat_graphic  TCAV=0.137  CAV acc=0.857  cos=-0.115+-0.076  p=0.55


## 7. Failure modes: are the blamed units actually load-bearing?

An attribution can be faithful in a deletion curve and still be irrelevant to the
decision. The sharper test: delete the top 10% of units the winner blames, recompute
the decision set from the released thresholds, and ask whether the label changed. Text
deletion flips the attention mask; image deletion writes the mean colour into whole
cells. Each is compared with random deletion of the same number of units, three
repeats, exact McNemar on the discordant pairs. n = 3000 analysis samples.

| ablation | flips | random | difference | McNemar p |
|---|---|---|---|---|
| text | 38.8% | 11.1% | +27.7 pp | 5.3e-254 |
| image | 0.2% | 0.1% | +0.1 pp | 6.9e-01 |
| both | 38.8% | 11.1% | +27.7 pp | 5.8e-257 |

The threshold is a 5 pp difference with p < 0.05. The random arm is reduced to its
majority vote across the three repeats before pairing, so each sample contributes one
binary random outcome against the single top-k outcome. Note what the comparison is:
top-10% text is about three tokens, top-10% image is about five cells, and on this
model removing three well-chosen tokens is a much more violent intervention than
removing five patch cells, which is the asymmetry §5 measured from the
coalition side.

Breaking the same flips down by error type, against the strict-majority label set:

| outcome | n | share | text | image | both |
|---|---|---|---|---|---|
| correct | 738 | 24.6% | 42.7% | 0.0% | 42.7% |
| false positive | 1956 | 65.2% | 33.7% | 0.2% | 33.7% |
| false negative | 1 | 0.0% | 0.0% | 0.0% | 0.0% |
| over trigger | 239 | 8.0% | 65.7% | 0.8% | 65.7% |
| misclass | 66 | 2.2% | 48.5% | 0.0% | 48.5% |

`over_trigger` means the model predicts at least one head the majority did not, with
no missed head; `misclass` means it swapped one class for another; `false_negative`
means it predicted nothing where the majority saw hate. The interesting question is
whether the explanations explain *errors* differently from successes. If the text
ablation flips confident correct cases as easily as it flips false negatives, the
explanation is describing the decision rule and not the mistake.

**What the ablation actually removes.** Forty cases were reviewed by hand: 20 text
flips and 20 controls matched on the error class (8 false positives, 7 correct, 4
over-trigger, 1 misclass per arm). Each case showed the deleted tokens highlighted and
the before/after head probabilities. The coding is descriptive: 20 per arm cannot
support a rate.

| deleted span contains | flips (n = 20) | share | controls (n = 20) | share |
|---|---|---|---|---|
| slur or group reference | 20 | 100% | 9 | 45% |
| URL / @mention / RT | 2 | 10% | 10 | 50% |
| ordinary content word | 2 | 10% | 5 | 25% |
| function word / punctuation | 0 | 0% | 1 | 5% |

Two readings. On every one of the 20 flips the deleted span contained a slur or group
reference (20/20), so the flips are not a ranking artefact that
happens to break the input: the method found the word that carries the label. On the
control arm the same top-10% ranking lands on a URL fragment in 10/20
cases and on a slur in only 9/20: when nothing flips, the deleted units are
frequently ones the model does not use. The contrast between the columns is the useful
part and it points the same way as the McNemar test.

The verdict field failed and is reported as such:
19/20 flips and 19/20 controls were marked
"justified". The question asked whether the *flip* was meaningful, which has no meaning
for a case that did not flip, so the two columns agreeing carries no information. A
better instrument would ask, for both arms, whether the deleted span is the span a human
would have deleted.

In [8]:
flip = load("p4_flip.json")
for name, block in flip["modalities"].items():
    print(f"{name:6s} top {block['top_flip_rate']:.3f}  random {block['random_flip_rate']:.3f}  "
          f"diff {block['difference']:+.4f}  McNemar p {block['mcnemar_p']:.2e}")

print()
table = pd.DataFrame(flip["outcomes"]).T
print(table[["n", "share", "text_flip_rate", "image_flip_rate", "both_flip_rate"]].round(3).to_string())
if "outcome_flip_table" in flip:
    block = flip["outcome_flip_table"]
    print()
    print(f"homogeneity of the both-modality flip rate across outcome classes: "
          f"chi2={block['chi2']:.2f}, p={block['p']:.2e}, groups={block['groups']}")

if os.path.exists(os.path.join(REFINE, "p4_qualitative.json")):
    qual = load("p4_qualitative.json")
    print()
    for group, counts in qual["groups"].items():
        print(f"{group:8s}", {k: v for k, v in counts.items() if k != "n"})
    print("share of flips whose deleted span is a slur:", qual["share_slur_flip"],
          "| controls:", qual["share_slur_control"])

text   top 0.388  random 0.111  diff +0.2768  McNemar p 5.31e-254
image  top 0.002  random 0.001  diff +0.0008  McNemar p 6.88e-01
both   top 0.388  random 0.111  diff +0.2771  McNemar p 5.75e-257

                     n  share  text_flip_rate  image_flip_rate  both_flip_rate
correct          738.0  0.246           0.427            0.000           0.427
false_positive  1956.0  0.652           0.337            0.002           0.337
false_negative     1.0  0.000           0.000            0.000           0.000
over_trigger     239.0  0.080           0.657            0.008           0.657
misclass          66.0  0.022           0.485            0.000           0.485

homogeneity of the both-modality flip rate across outcome classes: chi2=101.59, p=7.07e-22, groups=['correct', 'false_positive', 'over_trigger', 'misclass']

flip     {'justified': 19, 'tag:slur': 20, 'tag:negation': 1, 'unclear': 1, 'with_note': 1, 'tag:link': 2, 'tag:content': 2}
control  {'justified': 19, 'tag:slur': 9, 'w

## 8. Forcing the model to use the image: a modality-level ROAR

Removing an important component should hurt. Applied at the modality level that logic
becomes a statement about training rather than attribution: if text reliance is a
training artefact, retraining the fusion head with modality dropout should raise the
image's share without costing much accuracy. The encoders stay frozen and only the
fusion block is touched, so the arms differ in one thing.

| arm | eval macro-F1 | train macro-F1 | image share of $|\phi|$ | image dominance |
|---|---|---|---|---|
| 1 released head | 0.576 | --- | 6.5% | 0.5% |
| 2 re-initialised, refit on 2,000 | 0.461 ± 0.004 | 0.642 | 22.7% ± 9.6 | 15.0% |
| 3 same, plus modality dropout p = 0.3 | 0.483 ± 0.002 | 0.558 | 11.3% ± 0.5 | 7.1% |

Three seeds per arm, 400 epochs, stratified 2,000/1,000 split inside the analysis set
fixed across arms. Pre-registered: the image's share must rise while macro-F1 drops by
at most 0.01. The dropout arm moves the share by
-11.4 pp and macro-F1 by
+0.0224, so the criterion fails, and it fails in the wrong
direction. Modality dropout made the head rely on the image *less*, not more.

The mechanism is worth stating because it explains why the obvious fix does not work.
Dropout replaces the modality feature with its blank value but leaves the presence flag
at 1, so the head learns that *this particular feature vector* carries no information
rather than that *the image* should be weighted up. At evaluation the real image still
receives the same small weight. Buying image reliance needs the modality removed at the
input and the label preserved, which is a different training objective.

**What this comparison can and cannot support.** The refit arms see 2,000 samples and
reach an eval macro-F1 of 0.461 against 0.576 for the released head, so absolute
numbers are not comparable and arm 1 is a scale reference only. Arm 2 against arm 3
shares data, capacity and optimiser, so that contrast is the interpretable one. The sign
is stable across training budgets: at 150 / 400 / 800 epochs the image share moves
0.178→0.110, 0.227→0.113 and 0.206→0.120, and in every budget the dropout arm is the
*more accurate* one (+2.6 / +2.2 / +2.1 pp macro-F1). Under a balanced objective the head
does not merely fail to pick up the image; it does better by leaning on the text harder.
A test at the original training scale is not available here, and §12 says so. One
correction belongs here too: the first run of this experiment used an inverted class
weight in the BCE (n_pos/n_neg instead of n_neg/n_pos), which down-weighted the rare
heads and produced eval F1s near 0.38; the numbers above are from the corrected
objective (§11).

In [9]:
roar = load("p6_roar.json")
arms = pd.DataFrame(roar["arms"]).T[
    ["macro_f1", "train_macro_f1", "phi_i_share", "image_dominance"]]
print(arms.apply(pd.to_numeric, errors="coerce").round(4).fillna("---").to_string())
print()
print(json.dumps(roar["comparison"], indent=1))
print()
print("per-seed image share, arm2:", [round(v, 3) for v in roar["arms"]["arm2_refit"]["phi_i_share_per_seed"]])
print("per-seed image share, arm3:", [round(v, 3) for v in roar["arms"]["arm3_refit_dropout"]["phi_i_share_per_seed"]])

                    macro_f1 train_macro_f1  phi_i_share  image_dominance
arm1_released         0.5758            ---       0.0651           0.0050
arm2_refit            0.4608         0.6417       0.2266           0.1503
arm3_refit_dropout    0.4832          0.558       0.1127           0.0707

{
 "delta_phi_i_share": -0.11387687921524049,
 "delta_macro_f1": 0.022399768361284322,
 "criterion": "arm3 - arm2 phi_I share > 0 and macro-F1 drop <= 0.01",
 "passes": false
}

per-seed image share, arm2: [0.139, 0.33, 0.211]
per-seed image share, arm3: [0.119, 0.108, 0.112]


## 9. Does the model read the same cues as people?

If the model leaned on visual context as much as humans do, samples where annotators
agree should show a *smaller* text share than samples where they disagree: the model
would be using the picture precisely where the words are ambiguous. The pre-registered
primary analysis restricts to the hate-relevant subset, samples where at least one
annotator gave a usable hate label (`religion` does not count as one): n =
2,202 of 3,000, with 798 all-none tweets excluded, because
leaving them in turns the question into "normal tweets against hateful tweets". On that
subset, agreement (1.0 / 0.67 / 0.33 from the raw annotations) against the text share by
one-sided Spearman:

$$\rho = -0.071, \quad p_{\text{one-sided}} = 4.6e-04$$

The magnitude does not clear the -0.2 threshold by a wide margin, so the hypothesis is
reported as a miss (§11). It also gets weaker under its own controls: partialling out the
human usable-vote count and the $v(T,I)$ quartile leaves
$\rho_{\text{partial}} = -0.045$
(p = 0.017). On the full 3,000 the raw correlation is
-0.102, but under the same controls it is
-0.008. Most of the uncontrolled association is
agreement moving with the target level, not with the modality mix. The first
version of this analysis ran the full-sample correlation without the controls and read the
result as a directional near-miss; the restricted and controlled version is what is
reported here, and it is weaker.

| agreement bucket | n | mean text share | mean $\phi_T$ | mean target |
|---|---|---|---|---|
| all agree | 154 | 94.2% | +0.0850 | 0.420 |
| partial | 1571 | 93.5% | +0.0501 | 0.385 |
| all differ | 477 | 93.8% | +0.0896 | 0.424 |

Inside the subset the buckets are not monotone in $\phi_T$
(+0.0850 / +0.0501 /
+0.0896) and the text share barely moves
(94.2% to 93.8%),
which is consistent with the partial correlation: what changes with agreement is mostly
how *hateful* the sample is, not how the model splits that hate between modalities. Bucket
means stay descriptive; the Kruskal-Wallis statistic
(H = 17.1, p = 1.9e-04) cannot carry a
claim at three groups, and the pre-registered criterion remains the per-sample partial
correlation.

In [10]:
agreement = load("p5_agreement.json")
print(f"primary n={agreement['n']} rho={agreement['spearman_rho']:+.4f} "
      f"p1={agreement['spearman_p_one_sided_negative']:.2e} "
      f"partial={agreement['partial_rho']:+.4f} p1={agreement['partial_p_one_sided_negative']:.2e} "
      f"passes={agreement['passes']}")
print(f"full sample n={agreement['full_sample']['n']} rho={agreement['full_sample']['spearman_rho']:+.4f} "
      f"partial={agreement['full_sample']['partial_rho']:+.4f}")
print(pd.DataFrame(agreement["buckets"]).T[
    ["n", "text_share_mean", "phi_t_mean", "phi_i_mean", "target_mean"]].round(4).to_string())

primary n=2202 rho=-0.0705 p1=4.64e-04 partial=-0.0452 p1=1.69e-02 passes=False
full sample n=3000 rho=-0.1021 partial=-0.0080
                 n  text_share_mean  phi_t_mean  phi_i_mean  target_mean
all_agree    154.0           0.9415      0.0850      0.0014       0.4197
partial     1571.0           0.9353      0.0501      0.0014       0.3849
all_differ   477.0           0.9375      0.0896      0.0014       0.4243


## 10. Three checks that could have broken the story

**A3, preprocessing.** The tweet text is fed raw: URLs, @mentions and `RT` markers
included. Stripping them and recomputing the four coalitions gives
$\phi_T$ +0.0440 → +0.0454
(3.1% relative shift), rank correlation
+0.961, dominance 99.8% →
99.8%. The criterion was a shift below 10% with the
dominance ordering unchanged: pass. Truncated URLs and
@handles are not carrying the effect.

**Target sensitivity, four heads against five.** Dropping `religion` is a modelling
choice, so the whole analysis is recomputed with all five heads averaged:

| target | mean $\phi_T$ | mean $\phi_I$ | text dominance |
|---|---|---|---|
| four usable heads | +0.0467 | +0.0014 | 99.4% |
| five heads | +0.0267 | +0.0015 | 96.4% |

Dominance differs by 3.0 pp;
the ordering is unchanged, which is why the four-head target is defensible.

**A different checkpoint of the same idea (SigLIP2).** The H1 replication swaps
CLIP ViT-B/32 for `google/siglip2-base-patch16-224` with the same gated-fusion head and five heads,
on the same 3,000 analysis samples. Text is the larger modality on
94.8% of samples against
99.4% for the audited CLIP checkpoint, a gap of
4.6 pp against a pre-registered tolerance of 10 pp, so the
pattern is not specific to one set of weights.

Two caveats decide how much that sentence is worth. The absolute values are not
comparable: this fusion adds interaction terms, and both Shapley means come out negative
($\phi_T$ -0.0091, $\phi_I$ -0.0097), which
means the blank baseline outscores the real input on average for this checkpoint. Only
the share and the ordering are interpreted. And the released forward cannot run on
transformers 5 as shipped: it passes a `BaseModelOutputWithPooling` straight into
`F.normalize`. The fusion block is therefore reimplemented from the pooler tensors; a parity
check against the checkpoint's own forward gives a maximum absolute difference of 0.0 on
eight samples. The wording is "robust", not "reproduced": a different tokenizer,
thresholds and feature width make this a check that the pattern generalises, not a
replication of the same model.

In [11]:
robust = load("p8_robustness.json")
print(json.dumps(robust["a3_preprocessing"], indent=1))
print()
sens = pd.DataFrame({key: robust["target_sensitivity"][key]
                     for key in ("four_heads", "five_heads")}).T
print(sens[["phi_t_mean", "phi_i_mean", "text_dominance"]].round(4).to_string())
if os.path.exists(os.path.join(REFINE, "p7_siglip2_phi.json")):
    print()
    print(json.dumps(load("p7_siglip2_phi.json"), indent=1))

{
 "n": 499,
 "phi_t_mean_raw": 0.04404095560312271,
 "phi_t_mean_stripped": 0.04540642350912094,
 "relative_shift": 0.03100450150668621,
 "spearman_phi_t": 0.9607837110554535,
 "text_dominance_raw": 0.9979959919839679,
 "text_dominance_stripped": 0.9979959919839679,
 "criterion": "relative shift < 0.10 and dominance ordering unchanged",
 "passes": true
}

            phi_t_mean  phi_i_mean  text_dominance
four_heads      0.0467      0.0014          0.9943
five_heads      0.0267      0.0015          0.9643

{
 "n": 3000,
 "checkpoint": "Amirhossein75/siglip2-base-mmhs150k-fusion",
 "encoder": "google/siglip2-base-patch16-224",
 "thresholds": [
  0.3,
  0.75,
  0.85,
  0.2,
  0.55
 ],
 "phi_t": {
  "mean": -0.00905468501150608,
  "median": -0.026845037937164307
 },
 "phi_i": {
  "mean": -0.009727791883051395,
  "median": -0.009991295635700226
 },
 "text_dominance_share": 0.9483333333333334,
 "phi_i_negative_share": 0.9463333333333334,
 "main_text_dominance_share": 0.9943333333333333,
 "

## 11. What went wrong, including one bug that would have faked the result

Eleven negative or uncomfortable outcomes are reported here.

**1. A silent bug made three text methods identical.** The first P1.3 run produced
LIME, KernelSHAP and the random ordering with the *same* AOPC to six decimals. The
cause: `content_positions` expected a 1-D attention mask and returned an empty list for
the `(1, 77)` tensor the tokenizer actually produces, so every text method fell back to
in-order ranking and "did LIME beat random?" was measuring nothing. The tell was the
exact equality across methods; the whole file was deleted and regenerated after the
function was fixed to squeeze the batch dimension. Had it survived, §3 would have
reported that no text method beats random on a model that is 99.4% text-driven.

**2. The reference method is the worst text method.** Occlusion ranks below random on
text (+0.0013 versus +0.0179). Single-token
deletions are strongly non-additive on this checkpoint, so a leave-one-out ranking
disagrees with the model's own additive structure; LIME and KernelSHAP, which fit that
structure, win. This is a caution against treating occlusion as a neutral ground truth.

**3. The winner's metric is its own objective.** AOPC rewards exactly what LIME
optimises. The KernelSHAP margin of 0.0015 is small, and the
honest statement is that both surrogate methods clear the floor, not that LIME is
"the" faithful method.

**4. A pre-registered hypothesis missed, and the first version of it was the wrong
analysis.** On the pre-registered hate-relevant subset the correlation is
-0.071 against a required -0.20 (§9), and it shrinks to
-0.045 once the usable-vote count and the target level are controlled.
The number reported for two days was the full-sample correlation
(-0.102), which under the same controls collapses to
-0.008: most of it was agreement moving with the target level, not
with the modality mix. The larger number is significant and correctly signed, which makes it attractive;
the report keeps the restricted, controlled one and marks the hypothesis a miss.

**5. The concept probe does not work, for a reason unrelated to labels.** The per-sample gradients are 0.83-collinear, so the score saturates at 0 or 1 and the random-direction null does the same. The concept whose labels are demonstrably good (`printed_text` at 0.93/0.94) is also the one whose CAV only reaches 0.62 accuracy, which locates the failure in the gradient.
This is reported as a methodological failure rather than as evidence about the model.

**6. The intervention that should have worked did not.** Modality dropout moved the image share by -11.4 pp against a pre-registered positive direction, while macro-F1 moved +0.022; the dropout head is simultaneously less image-reliant and more accurate. The refit is
sample-limited (2,000 samples, §8), so the fair reading is that this cheap version of
the fix fails, not that the fix is impossible.

**7. The probe that should have supported the leakage story came back negative.**
The attribution inside printed-text boxes is -0.5 pp below an equal-area control (p = 2.1e-07), against a pre-registered +15 pp. The hypothesis that the image channel re-reads printed text is therefore
not supported by the one measurement designed to find it.

**8. The gold standard was missing for most of the project.** EasyOCR coverage was
reported for two days without any error estimate. A 200-image blind gold set fixed that
late (§1): detection 70.9%, false-positive rate 0%, and an implied text-bearing share
within 0.2 pp of the human rate. Any claim that X% of images contain Y needs its error rate measured
when the claim is made, not after it has propagated into three other sections.

**9. Detached runs died silently.** Two long evaluations exited with no traceback, no
error log and no Windows crash event, at roughly the same sample count. Re-running in
the foreground with `faulthandler` completed both times. The pipeline therefore writes
one JSON line per sample and resumes from the file, so a kill costs at most one sample;
every long script here is restartable for that reason.

**10. The first overlap control was geometrically larger than the thing it controlled
for.** The equal-area control sized its square from the *sum* of the OCR box areas while
the observed statistic used the *union* of those boxes; where boxes overlap the sum
exceeds the union (mean ratio 0.955, and below 0.98 for 46% of the 1,113 samples), so the
control square was too big and the excess looked more negative than it is. Recomputing
with a union-area control moves the excess from −1.16 pp to −0.48 pp (§6). The verdict is
unchanged: it still misses +15 pp in the negative direction. The earlier number
overstated the effect by a factor of 2.4, and both variants are now reported side by side.

**11. The first ROAR objective inverted the class weights.** `pos_weight` in the BCE was
set to n_pos/n_neg, which *down*-weights the rare heads (religion at 2.7% prevalence got
a weight near 0.1) and pushed both refit arms to an eval macro-F1 near 0.38. Corrected to
n_neg/n_pos the arms reach 0.461 and 0.483, and the comparison becomes sharper rather
than softer: dropout now costs nothing in accuracy while still cutting the image share
(§8). A wrong objective that makes both arms look bad is worse than a wrong objective
that makes one arm look bad, because it invites the reader to blame the sample size.

In [12]:
matrix = load("p1_matrix.json")
print("occlusion vs random on text:",
      f"{matrix['text']['occlusion']['aopc_del']['mean']:+.4f}",
      "vs", f"{matrix['text']['lime']['aopc_del']['random_mean']:+.4f}")
print("AOPC is LIME's own objective; the KernelSHAP gap is",
      f"{matrix['text']['lime']['aopc_del']['mean'] - matrix['text']['kernelshap']['aopc_del']['mean']:+.4f}")

gold = load("p0_text_gold.json")
print()
print(f"gold standard: n={gold['n_usable']} detection={gold['ocr_detection_rate']:.3f} "
      f"FPR={gold['ocr_false_positive_rate']:.3f} kappa={gold['agreement_kappa']:.3f}")

occlusion vs random on text: +0.0013 vs +0.0179
AOPC is LIME's own objective; the KernelSHAP gap is +0.0015

gold standard: n=199 detection=0.709 FPR=0.000 kappa=0.701


## 12. Conclusions

**The answer to the research question.** On `clip-vit-base-mmhs150k-mtl`, the text
modality accounts for 99.4% of the decision in a two-player
Shapley decomposition, with mean $\phi_T$ = +0.0467 against
$\phi_I$ = +0.0014. The image is used, but weakly and often
adversely: $\phi_I < 0$ on 56.5% of samples. Ablating the top 10% of blamed text units flips 38.8% of decisions against 11.1% for random units, which is what makes the attributions more than a description of the input.

**Which explainers can be trusted.** Only methods that were checked against deletion
and against a randomised model are used for the interpretation: LIME and KernelSHAP on
text, occlusion on image. Grad-CAM, integrated gradients and Score-CAM fail the image
floor on this checkpoint, and occlusion fails the text floor. Both facts are results
about the model-plus-method pair, not about the methods in general.

**What the image channel is not.** Three probes tried to give the small image
contribution a content, and all three came back negative. The image attribution inside printed-text boxes is -0.5 pp below an equal-area control, so the leakage hypothesis is not supported. The concept
probe cannot adjudicate: its gradients are 0.83-collinear so the score saturates, two of
its three concept sets are contaminated on the negative side, and the one clean concept
still yields a 0.62-accuracy CAV (§6). Re-fitting the fusion head with modality dropout moved the image's share by -11.4 pp while macro-F1 moved +0.0224: the head became less image-reliant and more accurate at once, so the cheap version of the fix is worse than useless.

**Why the gate cannot be the explanation.** The fused gate correlates with the Shapley
split at ρ = +0.45 (p = 5.1e-149):
statistically real, practically useless for a per-sample claim, and sign agreement with
$\phi_T$ is only 86.2%. A model card that reports gate
averages as evidence of multimodal use would be overclaiming.

**The measurement the dataset does not ship.** MMHS150K provides no OCR for this split,
and EasyOCR's raw coverage (36.6%) is a lower bound: a 200-image blind gold set puts the
detection rate at 70.9% with a 0% false-positive rate, and the implied true text-bearing
share (51.6%) matches the human rate (51.8%) to 0.2 pp. That number matters for anyone
who plans to study "text in images" here: roughly half the corpus contains readable
words, and the standard detector finds seven in ten of them.

**What would change the verdict.** A checkpoint trained with modality dropout on the
full training set, which this project could not run at scale (§8); and a dataset whose
images carry labels the text does not. Until then the defensible engineering
recommendation is to report the per-modality ablation next to the accuracy numbers,
because a 0.99 text share is invisible in a macro-F1 of 0.6536.

## 13. Reproduction

| stage | script | output |
|---|---|---|
| gate + metrics | `scripts/p0_gate.py` | `p0_gate.json`, `p0_baseline_probs.npz` |
| OCR boxes | `scripts/p0_ocr.py` | `p0_ocr.jsonl`, `p0_ocr_stats.json` |
| OCR gold standard (manual) | `scripts/build_annotation.py`, `scripts/merge_annotation.py` | `annotate/gold.html`, `p0_text_gold.json` |
| seven-method attributions, 499 samples | `scripts/p1_text_attributions.py`, `scripts/p1_image_attributions.py` | `p1_text_attr.jsonl`, `p1_image_attr.jsonl` |
| faithfulness scoring | `scripts/p1_evaluate.py` | `p1_faithfulness.jsonl`, `p1_matrix.json` |
| winner rule | `scripts/p1_decision.py` | `p1_decision.json` |
| gates, stability, agreement | `scripts/p1_sanity_stability.py` | `p1_sanity_curves.json`, `p1_stability.json`, `p1_agreement.json` |
| winner on 3,000 | `scripts/p1d_winner_3k.py` | `p1d_*_attr_test.jsonl` |
| modality Shapley | `scripts/p2_modality.py` | `p2_phi.jsonl`, `p2_curves.json`, `p2_summary.json` |
| concepts, TCAV, label audit | `scripts/p3_concepts.py`, `scripts/p3_tcav.py`, `scripts/merge_annotation.py` | `p3_concepts.json`, `p3_tcav.json`, `p3_concept_audit.json` |
| OCR overlay | `scripts/p3_overlap.py` | `p3_overlap.json` |
| failure modes | `scripts/p4_flip.py` | `p4_flip.json` |
| human agreement | `scripts/p5_agreement.py` | `p5_agreement.json` |
| ROAR | `scripts/p6_roar.py` | `p6_roar.json` |
| SigLIP2 replication | `scripts/p7_siglip2.py` | `p7_siglip2_phi.json` |
| robustness | `scripts/p8_robustness.py` | `p8_robustness.json` |
| fusion parity checks | `scripts/check_fusion_parity.py` | stdout (max abs delta, expected 0.0) |
| this notebook | `scripts/build_notebook.py`, `scripts/make_figures.py` | `Course Project-2026A.ipynb` |

Environment: Python 3.14, PyTorch 2.11 (CUDA 12.8) on an RTX 5080, transformers 5.17,
easyocr 1.7.2. Every long-running script appends one JSON line per sample and skips
completed ids on restart, so the pipeline is resumable at sample granularity. The
notebook itself only reads the stored artefacts; the one cell that touches the model is
the intervention demo in §2. The annotation pages were deployed to GitHub Pages at
`botherain.github.io/mmhs-annotation`; the three exports are kept in `annotate/out/` and
merged by `scripts/merge_annotation.py`, which takes the majority label per item when
several annotators labelled the same image.